# Dynamic-k Comparison Notebook

Run this notebook top to bottom in the online IDE. It does not require opening a terminal. The cells call the project runner from Python, compare the baseline methods against the latent regime dynamic-k method, and display the saved results and plots.

The default path is conservative: first run a tiny smoke comparison. After that succeeds, set `RUN_REAL = True` in the real-run cell to use the default Qwen models on the 40 GB GPU. Smoke and real outputs are saved to separate folders.

In [ ]:
from __future__ import annotations

import csv
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

# Set this only if your notebook kernel starts outside the repository.
REPO_ROOT_OVERRIDE = ""


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "main.py").exists() and (candidate / "research").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Set REPO_ROOT_OVERRIDE.")


REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root(Path.cwd())
SCRIPT = REPO_ROOT / "research" / "v.poponnikov" / "notebooks" / "dynamic_k_comparison.py"
RESULT_BASE_DIR = REPO_ROOT / "research" / "v.poponnikov" / "results"
PLOTS_BASE_DIR = REPO_ROOT / "research" / "v.poponnikov" / "plots"
RESULT_DIRS = {
    "smoke": RESULT_BASE_DIR / "smoke",
    "real": RESULT_BASE_DIR / "real",
}
PLOTS_DIRS = {
    "smoke": PLOTS_BASE_DIR / "smoke",
    "real": PLOTS_BASE_DIR / "real",
}
DISPLAY_RUN = "smoke"  # Change to "real" after the real run finishes.
EXPERIMENTS = ["01_baseline", "08_+speedup_adapt", "latent_regime_k"]

print(f"Repository: {REPO_ROOT}")
print(f"Comparison script: {SCRIPT}")

## Helper for Cell-Based Runs

This helper runs Python commands with `PYTHONPATH=src` and streams output into the notebook cell.

In [ ]:
def run_python(*args: object) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src")
    env.setdefault("HF_HOME", str(REPO_ROOT / ".hf-cache"))
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("USE_TF", "0")
    env.setdefault("USE_FLAX", "0")
    env.setdefault("TRANSFORMERS_NO_TF", "1")
    env.setdefault("TRANSFORMERS_NO_FLAX", "1")
    env.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

    command = [sys.executable, *[str(arg) for arg in args]]
    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")

## Optional Dependency Install

Run this once on a fresh online IDE image. It installs runtime dependencies only, skipping the incompatible `dev` extra on Python 3.10. It also forces a CUDA 12.4 PyTorch build, which is compatible with servers whose NVIDIA driver reports CUDA 12.5. The NumPy pin is intentionally `<2` because this online image has older compiled packages such as TensorFlow, numexpr, and bottleneck.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    run_python(
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
        "torch>=2.5,<2.11",
    )
    runtime_packages = [
        "typer>=0.15",
        "rich>=13",
        "tqdm>=4.68",
        "transformers>=4.48,<5",
        "accelerate>=1.3",
        "bitsandbytes>=0.45",
        "safetensors>=0.5",
        "datasets>=3,<5",
        "pandas>=2.2",
        "numpy<2",
        "scipy>=1.14",
        "scikit-learn>=1.6",
        "tokenizers>=0.21",
        "regex>=2024",
        "pyahocorasick>=2.3",
        "networkx>=3.4",
        "pydantic>=2.10",
        "python-dotenv>=1.2",
        "matplotlib>=3.10",
        "seaborn>=0.13",
    ]
    run_python("-m", "pip", "install", *runtime_packages)
    print("Dependency install finished. Restart the notebook kernel once, then rerun the setup/helper cells with INSTALL_DEPENDENCIES = False.")
else:
    print("Dependency install skipped.")

## GPU Check

In [ ]:
try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", props.name)
        print("VRAM GB:", round(props.total_memory / 1024**3, 2))
except ModuleNotFoundError:
    print("Torch is not installed in this kernel. Enable the dependency install cell above.")

## Confirm Research Experiments Are Discoverable

In [ ]:
run_python(REPO_ROOT / "src" / "main.py", "--list", "--research")

## Tiny Smoke Comparison

This runs the baseline, speedup-adaptive, and latent regime experiments with small OPT models. Use this first to check that the online IDE can download models, load datasets, and write plots.

In [ ]:
RUN_SMOKE = True
SMOKE_SAMPLES = 5
SMOKE_MAX_NEW_TOKENS = 32

if RUN_SMOKE:
    run_python(
        SCRIPT,
        "--output-dir",
        RESULT_DIRS["smoke"],
        "--plots-dir",
        PLOTS_DIRS["smoke"],
        "--tiny",
        "--samples",
        SMOKE_SAMPLES,
        "--max-new-tokens",
        SMOKE_MAX_NEW_TOKENS,
        "--device",
        "cuda",
    )
else:
    print("Smoke run skipped.")

## Real Qwen Comparison

After the smoke run succeeds, set `RUN_REAL = True` and run this cell. With 40 GB VRAM, the default 0.5B drafter plus 7B target in 4-bit should fit.

In [ ]:
RUN_REAL = True
REAL_SAMPLES = 50
REAL_MAX_NEW_TOKENS = 128

if RUN_REAL:
    run_python(
        SCRIPT,
        "--output-dir",
        RESULT_DIRS["real"],
        "--plots-dir",
        PLOTS_DIRS["real"],
        "--no-tiny",
        "--samples",
        REAL_SAMPLES,
        "--max-new-tokens",
        REAL_MAX_NEW_TOKENS,
        "--device",
        "cuda",
    )
else:
    print("Real run skipped. Set RUN_REAL = True when the smoke run is clean.")

## View Merged Results

In [ ]:
csv_path = RESULT_DIRS[DISPLAY_RUN] / "dynamic_k_comparison.csv"
print(f"Displaying {DISPLAY_RUN} results from: {csv_path}")
if not csv_path.exists():
    print(f"No merged CSV yet: {csv_path}")
else:
    try:
        import pandas as pd

        display(pd.read_csv(csv_path))
    except ModuleNotFoundError:
        with csv_path.open("r", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                print(row)

## View Plots

In [ ]:
plot_names = [
    "primary_metrics.png",
    "dynamic_k_summary.png",
    "dynamic_k_distribution.png",
    "controller_diagnostics.png",
]

print(f"Displaying {DISPLAY_RUN} plots from: {PLOTS_DIRS[DISPLAY_RUN]}")
for plot_name in plot_names:
    path = PLOTS_DIRS[DISPLAY_RUN] / plot_name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print(f"Missing plot: {path}")